### 基础版本（分块逐一回测再拼接，内存占用低，但是速度慢）

In [ ]:

if __name__ == '__main__':
    """
    开发调试专用模块：分块循环回测引擎
    (这部分代码保持不变，用于本地分段跑数据验证)
    """
    from bigmodule import M
    from datetime import datetime
    import pandas as pd
    import structlog
    import gc 

    logger = structlog.get_logger()
    # 注意：提交时平台会自动替换 datasource，但本地调试时需要手动指定
    datasource = 'cpt_dwc_2026_stock_hs300_snapshot'

    # ==========================================
    # 配置回测时间范围
    # ==========================================
    full_start_date = '2023-01-01'
    full_end_date = '2023-06-30'  # 示例：先跑一个月验证代码逻辑
    
    date_ranges = pd.date_range(start=full_start_date, end=full_end_date, freq='MS')
    all_results = [] 

    logger.info(f"🚀 Starting Momentum Factor Backtest: {full_start_date} to {full_end_date}")

    for start_dt in date_ranges:
        current_start = start_dt.strftime('%Y-%m-%d 00:00:00')
        current_end = (start_dt + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d 23:59:59')

        logger.info(f"Processing Chunk: {current_start} => {current_end}")

        try:
            df_chunk = main(datasource, current_start, current_end)
            
            if df_chunk is not None and not df_chunk.empty:
                all_results.append(df_chunk)
                logger.info(f"✅ Chunk Done. Rows: {len(df_chunk)}")
            else:
                logger.warning(f"⚠️ Chunk Empty: {current_start}")

            del df_chunk
            gc.collect() 
            
        except Exception as e:
            logger.error(f"❌ Error in chunk {current_start}: {e}")

    if all_results:
        logger.info("🧩 Concatenating all chunks...")
        final_data = pd.concat(all_results, ignore_index=True)
        final_data.sort_values(by=['date', 'instrument'], inplace=True)

        logger.info(f"🎉 All Done! Final Shape: {final_data.shape}")
        logger.info(f"Sample:\n{final_data.head()}")
        
        # 调用评估函数
        logger.info("📊 Starting Evaluation...")
        try:
            results = M.eval_dwc._latest(data=final_data)
        except Exception as e:
             logger.warning(f"Evaluation failed (local environment might miss specific modules): {e}")
             print("Data preview:", final_data.head())
    else:
        logger.error("No data generated.")

### 因子并行版（因子并行计算，回测统一完成）

## 经过测试，这是目前最快的版本！ 推荐使用这个！

In [ ]:
if __name__ == "__main__":
    """
    【并行版】开发调试专用模块：多线程分块循环回测引擎
    """
    from bigmodule import M
    import pandas as pd
    import structlog
    import gc
    # 引入并行库
    from concurrent.futures import ThreadPoolExecutor, as_completed

    logger = structlog.get_logger()
    datasource = "cpt_dwc_2026_stock_hs300_snapshot"

    full_start_date = "2023-01-01"
    full_end_date = "2024-12-01"

    # 1. 准备时间分片列表
    date_ranges = pd.date_range(start=full_start_date, end=full_end_date, freq="MS")
    chunk_params = []
    
    for start_dt in date_ranges:
        current_start = start_dt.strftime("%Y-%m-%d 00:00:00")
        current_end = (start_dt + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d 23:59:59")
        chunk_params.append((current_start, current_end))

    all_results = []
    
    # 2. 设置并行度 (关键!)
    # 建议: 4 (如果内存 < 32GB 建议设为 2 或 3)
    max_workers = 4 

    logger.info(f"🚀 Starting Parallel Backtest: {full_start_date} to {full_end_date}")
    logger.info(f"⚡ Parallel Workers: {max_workers}")

    # 3. 并行执行
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 提交所有任务
        # future 是一个未来的结果对象
        future_to_date = {
            executor.submit(main, datasource, start, end): (start, end) 
            for start, end in chunk_params
        }
        
        # 获取结果 (as_completed 会在任务完成时立即返回，不用按顺序等)
        for future in as_completed(future_to_date):
            start, end = future_to_date[future]
            try:
                df_chunk = future.result()
                
                if df_chunk is not None and not df_chunk.empty:
                    all_results.append(df_chunk)
                    logger.info(f"✅ Chunk Done: {start[:7]} | Rows: {len(df_chunk)}")
                else:
                    logger.warning(f"⚠️ Chunk Empty: {start[:7]}")
                
                # 显式清理局部变量
                del df_chunk
                gc.collect()

            except Exception as e:
                logger.error(f"❌ Error in chunk {start}: {e}")

    # 4. 拼接与评估 (保持不变)
    if all_results:
        logger.info("🧩 Concatenating all chunks...")
        final_data = pd.concat(all_results, ignore_index=True)
        # 记得重新排序，因为并行返回的顺序可能是乱的
        final_data.sort_values(by=["date", "instrument"], inplace=True)

        logger.info(f"🎉 All Done! Final Shape: {final_data.shape}")
        
        logger.info("📊 Starting Evaluation...")
        try:
            # 注意：如果数据量太大，eval_dwc 可能会耗时较久
            _ = M.eval_dwc._latest(data=final_data)
        except Exception as e:
            logger.warning(f"Evaluation failed: {e}")
            print("Data preview:", final_data.head())
    else:
        logger.error("No data generated.")

### 全部并行版（因子计算与回测均并行完成，速度较快，但是输出较混乱）

In [ ]:
if __name__ == "__main__":
    """
    【并行版】开发调试专用模块：多线程分块循环回测引擎
    """
    from bigmodule import M
    import pandas as pd
    import structlog
    import gc
    # 引入并行库
    from concurrent.futures import ThreadPoolExecutor, as_completed

    logger = structlog.get_logger()
    datasource = "cpt_dwc_2026_stock_hs300_snapshot"

    full_start_date = "2023-01-01"
    full_end_date = "2024-12-01"

    # 1. 准备时间分片列表
    date_ranges = pd.date_range(start=full_start_date, end=full_end_date, freq="MS")
    chunk_params = []
    
    for start_dt in date_ranges:
        current_start = start_dt.strftime("%Y-%m-%d 00:00:00")
        current_end = (start_dt + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d 23:59:59")
        chunk_params.append((current_start, current_end))

    all_results = []
    
    # 2. 设置并行度 (关键!)
    # 建议: 4 (如果内存 < 32GB 建议设为 2 或 3)
    max_workers = 4 

    logger.info(f"🚀 Starting Parallel Backtest: {full_start_date} to {full_end_date}")
    logger.info(f"⚡ Parallel Workers: {max_workers}")

    # 3. 并行执行
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 提交所有任务
        # future 是一个未来的结果对象
        future_to_date = {
            executor.submit(main, datasource, start, end): (start, end) 
            for start, end in chunk_params
        }
        
        # 获取结果 (as_completed 会在任务完成时立即返回，不用按顺序等)
        for future in as_completed(future_to_date):
            start, end = future_to_date[future]
            try:
                df_chunk = future.result()
                
                if df_chunk is not None and not df_chunk.empty:
                    all_results.append(df_chunk)
                    logger.info(f"✅ Chunk Done: {start[:7]} | Rows: {len(df_chunk)}")
                else:
                    logger.warning(f"⚠️ Chunk Empty: {start[:7]}")
                
                # 显式清理局部变量
                del df_chunk
                gc.collect()

            except Exception as e:
                logger.error(f"❌ Error in chunk {start}: {e}")

    # 4. 拼接与评估 (保持不变)
    if all_results:
        logger.info("🧩 Concatenating all chunks...")
        final_data = pd.concat(all_results, ignore_index=True)
        # 记得重新排序，因为并行返回的顺序可能是乱的
        final_data.sort_values(by=["date", "instrument"], inplace=True)

        logger.info(f"🎉 All Done! Final Shape: {final_data.shape}")
        
        logger.info("📊 Starting Evaluation...")
        try:
            # 注意：如果数据量太大，eval_dwc 可能会耗时较久
            _ = M.eval_dwc._latest(data=final_data)
        except Exception as e:
            logger.warning(f"Evaluation failed: {e}")
            print("Data preview:", final_data.head())
    else:
        logger.error("No data generated.")